In [0]:
# =============================================================================
# Notebook 09 — DFT Feature Enrichment
# Objective: Fetch DFT-computed properties for each compound in our dataset
#            from two open databases:
#              (1) Materials Project (mp-api) — band_gap, energy_above_hull,
#                  formation_energy_per_atom, is_stable, volume, density,
#                  spacegroup_number
#              (2) JARVIS-DFT (jarvis-tools, dft_3d) — optb88vdw_bandgap,
#                  mbj_bandgap, formation_energy_peratom, ehull
#
# Match strategy: reduced formula → if multiple hits, keep lowest energy_above_hull.
# Flag every row with dft_source (MP / JARVIS / none) and dft_match_quality
# (exact / reduced / none).
#
# The DFT layer (L4) is NEW and will be combined with L1+L3_te in NB10 to test
# whether physics-based ab-initio features fix the near-zero Spearman correlations
# seen in NB08.
#
# Inputs : ml_perovskites.master  (formula_reduced, base_compound, source_row_id)
# Outputs: ml_perovskites.X_dft_clean  (one row per source_row_id, DFT features)
#          ml_perovskites.dft_coverage  (coverage stats per database)
#          /tmp/09_dft_coverage.png
# Seed   : 42
# Author : Bernardo Araldi da Silva | UFSC | 2026
# =============================================================================

# =============================================================================
# SECTION 0 — Install dependencies (run this cell alone first)
# =============================================================================
%pip install mp-api jarvis-tools --quiet

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ----- Materials Project API key -----
# Set this in Databricks as a secret or environment variable:
#   os.environ["MP_API_KEY"] = "your_key_here"
# Get a free key at: https://materialsproject.org/api
MP_API_KEY = os.environ.get("MP_API_KEY", "")

if not MP_API_KEY:
    print("WARNING: MP_API_KEY not set. Materials Project queries will be skipped.")
    print("  Set it with: import os; os.environ['MP_API_KEY'] = 'your_key'")
    print("  Then re-run from Section 3 onward.")
else:
    print(f"MP_API_KEY found: {'*' * (len(MP_API_KEY) - 4) + MP_API_KEY[-4:]}")

print("Section 1 complete.")

# =============================================================================
# SECTION 2 — Load compound list from master
# =============================================================================

print("\n--- Loading compound list ---")

master_pd = spark.table("ml_perovskites.master").toPandas()

print(f"Master shape    : {master_pd.shape}")
print(f"Master columns  : {list(master_pd.columns)}")

# Determine which formula column to use (in priority order)
FORMULA_COL_CANDIDATES = [
    "formula_reduced",    # preferred: already reduced
    "base_compound",      # fallback: compound name, may need parsing
    "Compound_original",  # last resort: raw compound string
]
formula_col = next(
    (c for c in FORMULA_COL_CANDIDATES if c in master_pd.columns),
    None
)
if formula_col is None:
    raise ValueError(
        f"None of {FORMULA_COL_CANDIDATES} found in master. "
        f"Available columns: {list(master_pd.columns)}"
    )
print(f"Using formula column: '{formula_col}'")

# Determine which compound ID column to use
compound_id_col = next(
    (c for c in ["base_compound", "Compound_original", "source_row_id"]
     if c in master_pd.columns),
    "source_row_id"
)

keep_cols = list({c for c in ["source_row_id", formula_col, compound_id_col]
                  if c in master_pd.columns})

compounds_df = (master_pd[keep_cols]
                .drop_duplicates()
                .reset_index(drop=True)
                .rename(columns={formula_col: "formula_reduced"}))

unique_formulas = compounds_df["formula_reduced"].dropna().unique()

print(f"Total rows in master    : {len(master_pd)}")
print(f"Unique formula values   : {len(unique_formulas)}")
print(f"Sample formulas (first 10): {list(unique_formulas[:10])}")

# =============================================================================
# SECTION 3 — Query Materials Project
# =============================================================================

print("\n--- Querying Materials Project ---")

MP_FIELDS = [
    "material_id",
    "formula_pretty",
    "formula_anonymous",
    "energy_above_hull",
    "formation_energy_per_atom",
    "band_gap",
    "is_stable",
    "volume",
    "density",
    "nsites",
    "spacegroup",
    "symmetry",
    "theoretical",
]

mp_records = []

if MP_API_KEY:
    try:
        from mp_api.client import MPRester

        with MPRester(MP_API_KEY) as mpr:
            # Query all ABO3-type compounds once (all elements, O-containing, 3-element)
            # Then filter client-side to match our formula list
            print("  Fetching ABO3 candidates from MP (formula=**O3, 3-element)...")

            # Batch query: chemsys pattern for 3-element oxide
            # MP supports wildcard element queries via elements + num_elements
            docs = mpr.materials.summary.search(
                num_elements=[3],
                elements=["O"],
                fields=["material_id", "formula_pretty", "energy_above_hull",
                        "formation_energy_per_atom", "band_gap", "is_stable",
                        "volume", "density", "nsites", "symmetry", "theoretical"],
            )
            print(f"  MP returned {len(docs)} 3-element oxide candidates")

            for doc in docs:
                try:
                    sg_num = (doc.symmetry.number
                              if doc.symmetry is not None else None)
                    mp_records.append({
                        "mp_material_id":              doc.material_id,
                        "mp_formula_pretty":           doc.formula_pretty,
                        "mp_energy_above_hull":        doc.energy_above_hull,
                        "mp_formation_energy_per_atom":doc.formation_energy_per_atom,
                        "mp_band_gap":                 doc.band_gap,
                        "mp_is_stable":                int(bool(doc.is_stable)),
                        "mp_volume":                   doc.volume,
                        "mp_density":                  doc.density,
                        "mp_nsites":                   doc.nsites,
                        "mp_spacegroup_number":        sg_num,
                        "mp_theoretical":              int(bool(doc.theoretical)),
                    })
                except Exception as e:
                    continue

        mp_df = pd.DataFrame(mp_records)
        print(f"  MP records parsed: {len(mp_df)}")

    except ImportError:
        print("  mp-api not available. Skipping MP queries.")
        mp_df = pd.DataFrame()
    except Exception as e:
        print(f"  MP query error: {e}")
        print("  Continuing without MP data.")
        mp_df = pd.DataFrame()
else:
    print("  Skipping MP (no API key).")
    mp_df = pd.DataFrame()

# =============================================================================
# SECTION 4 — Load JARVIS-DFT (dft_3d dataset)
# =============================================================================

print("\n--- Loading JARVIS-DFT (dft_3d) ---")

jarvis_df = pd.DataFrame()

try:
    from jarvis.db.figshare import data as jarvis_data

    print("  Downloading dft_3d dataset (~200 MB, cached after first run)...")
    dft_3d = jarvis_data("dft_3d")
    print(f"  dft_3d records: {len(dft_3d)}")

    # Inspect available keys
    if dft_3d:
        sample_keys = list(dft_3d[0].keys())
        print(f"  Available keys (first entry): {sample_keys[:30]}")

        # Identify band gap and formation energy field names
        gap_keys  = [k for k in sample_keys if "gap" in k.lower()]
        form_keys = [k for k in sample_keys if "form" in k.lower() or "hull" in k.lower()]
        print(f"  Gap-related keys   : {gap_keys}")
        print(f"  Formation/hull keys: {form_keys}")

    jarvis_records = []
    for entry in dft_3d:
        try:
            formula = entry.get("formula", "")
            if not formula:
                continue
            jarvis_records.append({
                "jarvis_jid":                   entry.get("jid", ""),
                "jarvis_formula":               formula,
                "jarvis_optb88vdw_bandgap":     entry.get("optb88vdw_bandgap", np.nan),
                "jarvis_mbj_bandgap":           entry.get("mbj_bandgap", np.nan),
                "jarvis_formation_energy_atom": entry.get("formation_energy_peratom", np.nan),
                "jarvis_ehull":                 entry.get("ehull", np.nan),
                "jarvis_spacegroup_number":     entry.get("spg_number", np.nan),
                "jarvis_n_atoms":               entry.get("natoms", np.nan),
            })
        except Exception:
            continue

    jarvis_df = pd.DataFrame(jarvis_records)
    print(f"  JARVIS records parsed: {len(jarvis_df)}")
    print(f"  Non-null mbj_bandgap : {jarvis_df['jarvis_mbj_bandgap'].notna().sum()}")
    print(f"  Non-null ehull       : {jarvis_df['jarvis_ehull'].notna().sum()}")

except ImportError:
    print("  jarvis-tools not available. Skipping JARVIS queries.")
except Exception as e:
    print(f"  JARVIS error: {e}")
    print("  Continuing without JARVIS data.")

# =============================================================================
# SECTION 5 — Formula normalization utilities
# =============================================================================

print("\n--- Building formula normalization ---")

from pymatgen.core import Composition
from math import gcd
from functools import reduce

def reduce_formula(formula_str):
    """
    Convert a formula string to its reduced (anonymous) form using pymatgen.
    Returns the reduced formula string, or None on parse error.
    E.g.: 'BaTiO3' -> 'BaTiO3' (already reduced)
          'Ba2Ti2O6' -> 'BaTiO3'
    """
    try:
        comp = Composition(formula_str)
        # Get integer composition and compute GCD
        el_amt = {str(el): int(round(amt)) for el, amt in comp.items()}
        if not el_amt:
            return None
        g = reduce(gcd, el_amt.values())
        reduced = {el: amt // g for el, amt in el_amt.items()}
        # Rebuild sorted formula string (alphabetical — consistent key)
        return "".join(f"{el}{'' if amt == 1 else amt}"
                       for el, amt in sorted(reduced.items()))
    except Exception:
        return None

def canonical_formula(formula_str):
    """Return canonical reduced formula for matching."""
    r = reduce_formula(formula_str)
    return r.lower() if r else None

# Build lookup key for our dataset
compounds_df["formula_key"] = compounds_df["formula_reduced"].apply(canonical_formula)
print(f"  Formula keys built: {compounds_df['formula_key'].notna().sum()} / {len(compounds_df)}")

# =============================================================================
# SECTION 6 — Match with Materials Project
# =============================================================================

print("\n--- Matching with Materials Project ---")

mp_matched = {}  # formula_key -> best MP record (lowest energy_above_hull)

if not mp_df.empty:
    mp_df["formula_key"] = mp_df["mp_formula_pretty"].apply(canonical_formula)
    mp_df_valid = mp_df.dropna(subset=["formula_key"])

    for fkey in compounds_df["formula_key"].dropna().unique():
        hits = mp_df_valid[mp_df_valid["formula_key"] == fkey]
        if hits.empty:
            continue
        # Select lowest energy_above_hull as best structural match
        best = (hits.sort_values("mp_energy_above_hull", na_position="last")
                .iloc[0].to_dict())
        mp_matched[fkey] = best

    n_mp_hits = len(mp_matched)
    n_total   = len(compounds_df["formula_key"].dropna().unique())
    print(f"  Matched: {n_mp_hits} / {n_total} unique formulas "
          f"({n_mp_hits/n_total*100:.1f}%)")
else:
    print("  No MP data to match.")
    n_mp_hits = 0

# =============================================================================
# SECTION 7 — Match with JARVIS-DFT
# =============================================================================

print("\n--- Matching with JARVIS-DFT ---")

jarvis_matched = {}  # formula_key -> best JARVIS record

if not jarvis_df.empty:
    jarvis_df["formula_key"] = jarvis_df["jarvis_formula"].apply(canonical_formula)
    jarvis_df_valid = jarvis_df.dropna(subset=["formula_key"])

    for fkey in compounds_df["formula_key"].dropna().unique():
        hits = jarvis_df_valid[jarvis_df_valid["formula_key"] == fkey]
        if hits.empty:
            continue
        # Select lowest ehull as best structural match
        best = (hits.sort_values("jarvis_ehull", na_position="last")
                .iloc[0].to_dict())
        jarvis_matched[fkey] = best

    n_jarvis_hits = len(jarvis_matched)
    n_total       = len(compounds_df["formula_key"].dropna().unique())
    print(f"  Matched: {n_jarvis_hits} / {n_total} unique formulas "
          f"({n_jarvis_hits/n_total*100:.1f}%)")
else:
    print("  No JARVIS data to match.")
    n_jarvis_hits = 0

# =============================================================================
# SECTION 8 — Build DFT feature matrix
# =============================================================================

print("\n--- Building DFT feature matrix ---")

# DFT features to include (prefer MP values; fill gaps with JARVIS)
DFT_FEATURE_COLS = [
    # From MP
    "mp_energy_above_hull",        # thermodynamic stability (eV/atom)
    "mp_formation_energy_per_atom",# formation energy (eV/atom)
    "mp_band_gap",                 # PBE band gap (eV) — often underestimated
    "mp_is_stable",                # binary stability flag
    "mp_volume",                   # unit cell volume (Å³)
    "mp_density",                  # density (g/cm³)
    "mp_nsites",                   # atoms per unit cell
    "mp_spacegroup_number",        # crystal symmetry
    "mp_theoretical",              # flag: not in ICSD
    # From JARVIS (more accurate band gaps)
    "jarvis_optb88vdw_bandgap",    # OptB88vdW functional band gap (eV)
    "jarvis_mbj_bandgap",          # TBmBJ band gap — more accurate for oxides
    "jarvis_formation_energy_atom",# formation energy (eV/atom)
    "jarvis_ehull",                # energy above hull (eV/atom)
    "jarvis_spacegroup_number",    # crystal symmetry (double-check vs MP)
    "jarvis_n_atoms",              # atoms per unit cell
    # Derived
    "dft_bandgap_best",            # mbj_bandgap if available, else mp_band_gap
    "dft_ehull_best",              # jarvis_ehull if available, else mp_energy_above_hull
    "dft_formation_best",          # jarvis if available, else mp
    "dft_source",                  # MP / JARVIS_only / none
    "dft_n_sources",               # 0, 1, or 2 (how many DBs matched)
]

dft_rows = []

for _, row in master_pd.iterrows():
    src_id  = row["source_row_id"]
    # Use whichever formula column was identified in Section 2
    formula = row.get(formula_col, None)
    fkey    = canonical_formula(str(formula)) if pd.notna(formula) and formula else None

    rec = {"source_row_id": src_id, "formula_key": fkey}

    mp_rec     = mp_matched.get(fkey, {})     if fkey else {}
    jarvis_rec = jarvis_matched.get(fkey, {}) if fkey else {}

    # --- MP features ---
    rec["mp_energy_above_hull"]         = mp_rec.get("mp_energy_above_hull", np.nan)
    rec["mp_formation_energy_per_atom"] = mp_rec.get("mp_formation_energy_per_atom", np.nan)
    rec["mp_band_gap"]                  = mp_rec.get("mp_band_gap", np.nan)
    rec["mp_is_stable"]                 = mp_rec.get("mp_is_stable", np.nan)
    rec["mp_volume"]                    = mp_rec.get("mp_volume", np.nan)
    rec["mp_density"]                   = mp_rec.get("mp_density", np.nan)
    rec["mp_nsites"]                    = mp_rec.get("mp_nsites", np.nan)
    rec["mp_spacegroup_number"]         = mp_rec.get("mp_spacegroup_number", np.nan)
    rec["mp_theoretical"]               = mp_rec.get("mp_theoretical", np.nan)
    rec["mp_material_id"]               = mp_rec.get("mp_material_id", None)

    # --- JARVIS features ---
    rec["jarvis_optb88vdw_bandgap"]     = jarvis_rec.get("jarvis_optb88vdw_bandgap", np.nan)
    rec["jarvis_mbj_bandgap"]           = jarvis_rec.get("jarvis_mbj_bandgap", np.nan)
    rec["jarvis_formation_energy_atom"] = jarvis_rec.get("jarvis_formation_energy_atom", np.nan)
    rec["jarvis_ehull"]                 = jarvis_rec.get("jarvis_ehull", np.nan)
    rec["jarvis_spacegroup_number"]     = jarvis_rec.get("jarvis_spacegroup_number", np.nan)
    rec["jarvis_n_atoms"]               = jarvis_rec.get("jarvis_n_atoms", np.nan)
    rec["jarvis_jid"]                   = jarvis_rec.get("jarvis_jid", None)

    # --- Derived best-estimate features ---
    mbj = rec["jarvis_mbj_bandgap"]
    opt = rec["jarvis_optb88vdw_bandgap"]
    mpg = rec["mp_band_gap"]
    # Best band gap: TBmBJ > OptB88vdW > PBE (MP)
    if pd.notna(mbj) and mbj >= 0:
        rec["dft_bandgap_best"] = mbj
    elif pd.notna(opt) and opt >= 0:
        rec["dft_bandgap_best"] = opt
    elif pd.notna(mpg) and mpg >= 0:
        rec["dft_bandgap_best"] = mpg
    else:
        rec["dft_bandgap_best"] = np.nan

    jeh = rec["jarvis_ehull"]
    meh = rec["mp_energy_above_hull"]
    rec["dft_ehull_best"] = jeh if pd.notna(jeh) else meh

    jfe = rec["jarvis_formation_energy_atom"]
    mfe = rec["mp_formation_energy_per_atom"]
    rec["dft_formation_best"] = jfe if pd.notna(jfe) else mfe

    n_sources = int(bool(mp_rec)) + int(bool(jarvis_rec))
    rec["dft_n_sources"] = n_sources
    if bool(mp_rec) and bool(jarvis_rec):
        rec["dft_source"] = "MP+JARVIS"
    elif bool(mp_rec):
        rec["dft_source"] = "MP"
    elif bool(jarvis_rec):
        rec["dft_source"] = "JARVIS"
    else:
        rec["dft_source"] = "none"

    dft_rows.append(rec)

dft_pd = pd.DataFrame(dft_rows)

# Coverage summary
coverage = dft_pd["dft_source"].value_counts()
print(f"\n  DFT source coverage ({len(dft_pd)} rows):")
print(coverage.to_string())
pct_covered = (dft_pd["dft_source"] != "none").mean() * 100
print(f"\n  Overall coverage : {pct_covered:.1f}%")
print(f"  dft_bandgap_best : {dft_pd['dft_bandgap_best'].notna().mean()*100:.1f}% non-null")
print(f"  dft_ehull_best   : {dft_pd['dft_ehull_best'].notna().mean()*100:.1f}% non-null")
print(f"  dft_formation_best: {dft_pd['dft_formation_best'].notna().mean()*100:.1f}% non-null")

# =============================================================================
# SECTION 9 — Coverage plot
# =============================================================================

print("\n--- Generating coverage plot ---")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Source breakdown
src_counts = dft_pd["dft_source"].value_counts()
colors_src = {"MP+JARVIS":"#2ca02c","MP":"#1f77b4","JARVIS":"#ff7f0e","none":"#d62728"}
bars_colors = [colors_src.get(s, "#888") for s in src_counts.index]
axes[0].bar(src_counts.index, src_counts.values, color=bars_colors)
axes[0].set_xlabel("DFT source")
axes[0].set_ylabel("Row count")
axes[0].set_title("DFT coverage by source (per row)")
for i, (v, label) in enumerate(zip(src_counts.values, src_counts.index)):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=9)

# Band gap distribution: experimental (y_targets.bandgap_eV) vs DFT best
y_pd = spark.table("ml_perovskites.y_targets").toPandas()
bg_exp = y_pd["bandgap_eV"].dropna()

# Align dft_pd to y_pd on source_row_id
dft_al = dft_pd.merge(y_pd[["source_row_id"]], on="source_row_id", how="inner")
bg_dft = dft_al["dft_bandgap_best"].dropna()

axes[1].hist(bg_exp, bins=40, alpha=0.6, color="#1f77b4", label=f"Experimental (n={len(bg_exp)})")
axes[1].hist(bg_dft, bins=40, alpha=0.6, color="#d62728", label=f"DFT best (n={len(bg_dft)})")
axes[1].set_xlabel("Band gap (eV)")
axes[1].set_ylabel("Count")
axes[1].set_title("Band gap: experimental vs DFT")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/09_dft_coverage.png", dpi=130, bbox_inches="tight")
plt.close()
print("  Saved: /tmp/09_dft_coverage.png")

# =============================================================================
# SECTION 10 — Spearman correlation check: DFT features vs H2 rate
# =============================================================================

print("\n--- Spearman correlation: DFT features vs log1p H2 rate ---")

from scipy.stats import spearmanr

dft_feat_cols = [
    "mp_energy_above_hull", "mp_formation_energy_per_atom", "mp_band_gap",
    "mp_is_stable", "mp_volume", "mp_density", "mp_nsites",
    "mp_spacegroup_number", "jarvis_optb88vdw_bandgap", "jarvis_mbj_bandgap",
    "jarvis_formation_energy_atom", "jarvis_ehull",
    "dft_bandgap_best", "dft_ehull_best", "dft_formation_best",
]

dft_corr_df = dft_pd.merge(
    y_pd[["source_row_id", "log1p_h2_rate", "h2_rate_umol_g_h"]],
    on="source_row_id", how="inner"
)

active_mask = dft_corr_df["h2_rate_umol_g_h"] > 0
corr_rows = []
for col in dft_feat_cols:
    valid = dft_corr_df[active_mask][col].notna()
    n_valid = valid.sum()
    if n_valid < 20:
        corr_rows.append({"feature": col, "spearman_rho": np.nan, "n": n_valid})
        continue
    rho, _ = spearmanr(
        dft_corr_df[active_mask][col][valid],
        dft_corr_df[active_mask]["log1p_h2_rate"][valid]
    )
    corr_rows.append({"feature": col, "spearman_rho": rho, "n": n_valid})

corr_dft = (pd.DataFrame(corr_rows)
            .dropna()
            .sort_values("spearman_rho", key=abs, ascending=False))

print(f"  DFT feature correlations (active subset):")
print(corr_dft.to_string(index=False))
print(f"\n  Max |ρ| (DFT features): {corr_dft['spearman_rho'].abs().max():.3f}")
print(f"  Max |ρ| (L1 features, NB08): 0.149")

# =============================================================================
# SECTION 11 — Write X_dft_clean to Delta table
# =============================================================================

print("\n--- Writing X_dft_clean to Delta table ---")

# Drop non-numeric meta columns before writing feature table
non_feature_meta = ["formula_key", "mp_material_id", "jarvis_jid", "dft_source"]

# Keep dft_source as categorical for metadata; drop formula_key and IDs
dft_to_write = dft_pd.drop(columns=["formula_key", "mp_material_id", "jarvis_jid"])

# Encode dft_source as integer for feature use
source_map = {"none": 0, "MP": 1, "JARVIS": 2, "MP+JARVIS": 3}
dft_to_write["dft_source_code"] = dft_to_write["dft_source"].map(source_map).fillna(0).astype(int)

dft_spark = spark.createDataFrame(
    dft_to_write.where(pd.notnull(dft_to_write), other=None)
)
(dft_spark
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.X_dft_clean"))
n_dft = spark.table("ml_perovskites.X_dft_clean").count()
print(f"  ml_perovskites.X_dft_clean: {n_dft} rows")

# Coverage stats table
cov_rows = []
for src in ["MP", "JARVIS", "MP+JARVIS", "none"]:
    n = (dft_pd["dft_source"] == src).sum()
    cov_rows.append({"dft_source": src, "n_rows": int(n),
                     "pct": round(n / len(dft_pd) * 100, 1)})
cov_spark = spark.createDataFrame(pd.DataFrame(cov_rows))
(cov_spark
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("ml_perovskites.dft_coverage"))
print(f"  ml_perovskites.dft_coverage: written")

# =============================================================================
# SECTION 12 — Final summary
# =============================================================================

print("\n" + "="*60)
print("NOTEBOOK 09 COMPLETE")
print("="*60)
print(f"\nDFT databases queried:")
print(f"  Materials Project : {'YES' if MP_API_KEY else 'SKIPPED (no API key)'}")
print(f"  JARVIS-DFT dft_3d : {'YES' if not jarvis_df.empty else 'FAILED or empty'}")
print(f"\nCoverage (rows):")
for _, r in pd.DataFrame(cov_rows).iterrows():
    print(f"  {r['dft_source']:12s}: {r['n_rows']:4d} rows ({r['pct']:.1f}%)")
print(f"\nSpearman ρ — DFT vs log1p H2 rate (active rows):")
print(f"  Max |ρ| (DFT) : {corr_dft['spearman_rho'].abs().max():.3f}")
print(f"  Max |ρ| (L1)  : 0.149  (NB08 baseline)")
if corr_dft['spearman_rho'].abs().max() > 0.20:
    print(f"  DFT features add signal beyond L1 — proceed to NB10 (L1+L3_te+L4_dft).")
else:
    print(f"  DFT signal marginal — NB10 will test, but may need dataset expansion.")

print(f"\nOutput table : ml_perovskites.X_dft_clean ({n_dft} rows)")
print(f"\nNext step: NB10 — combine L1 + L3_te + L4_dft, rerun GroupKFold CV,")
print(f"  compare R² across all feature layers in one table.")

print(f"\n--- IMPORTANT: Materials Project API key ---")
if not MP_API_KEY:
    print(f"  MP was skipped. To include MP features:")
    print(f"  1. Get a free API key at https://materialsproject.org/api")
    print(f"  2. In Databricks: import os; os.environ['MP_API_KEY'] = 'YOUR_KEY'")
    print(f"  3. Re-run this notebook from Section 3 onward.")
    print(f"  JARVIS data alone may be sufficient if coverage is high.")